# TradeSage Large-Scale Training (10 Years)
This notebook downloads 10 years of historical data using `yfinance` and trains the memory-optimized XGBoost model.
By using `yfinance`, we bypass the Angel One API historical data limits.

In [ ]:
!pip install yfinance ta xgboost pandas scikit-learn tqdm

## 1. Fetch 10 Years of Data
This will download daily data for the top Indian stocks. This step takes 10-15 minutes.

In [ ]:
import os
import json
import yfinance as yf
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def fetch_yfinance_data(symbols, years=10, max_workers=10, cache_dir='data_cache_yfinance'):
    os.makedirs(cache_dir, exist_ok=True)
    yf_symbols = [f"{s}.NS" for s in symbols]
    results, failed = {}, []
    period = f"{years}y"

    def fetch_single(symbol):
        base_symbol = symbol.replace('.NS', '')
        cache_file = os.path.join(cache_dir, f"{base_symbol}_daily.csv")
        if os.path.exists(cache_file):
            try:
                df = pd.read_csv(cache_file, index_col='timestamp', parse_dates=True)
                if len(df) > 200: return base_symbol, df
            except: pass
        try:
            ticker = yf.Ticker(symbol)
            df = ticker.history(period=period)
            if df.empty or len(df) < 200: return base_symbol, None
            df.reset_index(inplace=True)
            time_col = 'Date' if 'Date' in df.columns else 'Datetime'
            df.rename(columns={time_col: 'timestamp', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume'}, inplace=True)
            df.set_index('timestamp', inplace=True)
            df = df[['open', 'high', 'low', 'close', 'volume']]
            df.to_csv(cache_file)
            return base_symbol, df
        except:
            return base_symbol, None

    print(f"\n\n📥 Fetching {years} years of data for {len(symbols)} stocks from yfinance...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_symbol = {executor.submit(fetch_single, s): s for s in yf_symbols}
        for future in tqdm(as_completed(future_to_symbol), total=len(yf_symbols), desc="Downloading"):
            symbol, df = future.result()
            if df is not None: results[symbol] = df
            else: failed.append(symbol)
    print(f"✓ Successfully fetched data for {len(results)} stocks. (Failed: {len(failed)})")
    return results

# Read symbols
json_path = os.path.join('data', 'nse_top_3000_angel.json')
symbols = json.load(open(json_path, 'r')) if os.path.exists(json_path) else ['RELIANCE', 'TCS', 'HDFCBANK']

# Run Fetch
stock_data = fetch_yfinance_data(symbols, years=10, max_workers=10)

## 2. Train Memory-Optimized Model
Reads all CSVs, builds Technical Indicators, strictly downcasts floats to prevent RAM saturation (`MemoryError`), sweeps garbage, and trains the model.

In [ ]:
import sys
import gc
from pathlib import Path
from datetime import datetime

# Adding local project root to path so we can import src folders in Google Colab
PROJECT_ROOT = os.path.abspath('.')
sys.path.append(PROJECT_ROOT)

from src.core.feature_engineering import FeatureEngineer
from src.core.model_training import TradingModelTrainer

cache_dir = 'data_cache_yfinance'
model_path = 'models/tradesage_10y_colab.pkl'

cache_path = Path(PROJECT_ROOT) / cache_dir
model_out = Path(PROJECT_ROOT) / model_path

print("="*80)
print(f" TRADESAGE COLAB LARGE-SCALE TRAINING ({cache_dir})")
print("="*80)

data_files = list(cache_path.glob('*_daily.csv'))
if not data_files:
    print(f"Cache directory {cache_dir} is empty! Run Cell 1 first.")
else:
    print(f"Found {len(data_files)} cached stocks. Loading and engineering features...")
    engineer = FeatureEngineer()
    all_data = []
    
    for file in data_files:
        symbol = file.stem.replace('_daily', '')
        try:
            df = pd.read_csv(file, index_col='timestamp', parse_dates=True)
            if df.index.tz is not None: df.index = df.index.tz_localize(None)
            if len(df) < 200: continue
                
            df_features = engineer.add_technical_indicators(df)
            df_final = engineer.create_target_variable(df_features, forward_days=10, gain_threshold=0.05)
            df_final['symbol'] = symbol
            all_data.append(df_final)
        except:
            pass
            
    print(f"\nSuccessfully engineered features for {len(all_data)} stocks.")
    print("Splitting each stock 80/20 chronologically (Train/Val)...")
    
    train_parts, val_parts = [], []
    for df in all_data:
        n = len(df)
        sp = int(n * 0.8)
        train_parts.append(df.iloc[:sp])
        val_parts.append(df.iloc[sp:])
        
    stocks_trained_count = len(all_data)
    del all_data
    gc.collect() # <--- Sweeping Garbage to prevent OOM
    
    train_df = pd.concat(train_parts)
    val_df   = pd.concat(val_parts)
    del train_parts, val_parts
    gc.collect() # <--- Sweeping Garbage to prevent OOM
    
    print("Downcasting float64 to float32 to save 50% RAM... (crucial for 3M+ rows)")
    for col in train_df.select_dtypes(include=['float64']).columns:
        train_df[col] = train_df[col].astype('float32')
        if col in val_df.columns:
            val_df[col] = val_df[col].astype('float32')
            
    X_train, y_train, feature_names = engineer.prepare_training_data(train_df)
    del train_df
    gc.collect()
    
    X_val, y_val, _ = engineer.prepare_training_data(val_df)
    del val_df
    gc.collect()
    
    for col in set(feature_names) - set(X_val.columns):
        X_val[col] = 0
    X_val = X_val[feature_names]
    
    print(f"Features: {len(feature_names)}")
    print(f"Train: {len(X_train):,} samples  (pos={y_train.mean()*100:.1f}%)")
    print(f"Val:   {len(X_val):,} samples  (pos={y_val.mean()*100:.1f}%)")
    
    # Train
    trainer = TradingModelTrainer()
    trainer.train_model(X_train, y_train, X_val, y_val)
    
    # Evaluate
    print("\nEvaluating model...")
    metrics = trainer.evaluate_model(X_val, y_val)
    
    # Save Output
    os.makedirs(model_out.parent, exist_ok=True)
    trainer.save_model(str(model_out))
    
    # Build JSON Report
    report_file = str(model_out).replace('.pkl', '_report.json')
    try:
        top_features = trainer.get_feature_importance(top_n=15).head(15).to_dict('records')
    except:
        top_features = []
        
    with open(report_file, 'w') as f:
        json.dump({
            'training_date': datetime.now().isoformat(),
            'stocks_trained': stocks_trained_count,
            'total_samples': len(X_train) + len(X_val),
            'features': len(feature_names),
            'metrics': {k: float(v) if not isinstance(v, list) else v for k, v in metrics.items() if k != 'confusion_matrix'},
            'top_features': top_features
        }, f, indent=2)
        
    print(f"Training report saved to {report_file}")
    print("COLAB TRAINING COMPLETE!")